# EloSense v3 — Honest Evaluation & Beyond

v1 and v2 both used a plain random `train_test_split(..., stratify=y)` with no grouping by player. This notebook does not touch `elosense_v1.ipynb` or `elosense_v2.ipynb` — it re-derives an honest baseline and builds on top of it.

**W1 goal**: check whether v1/v2's random split leaks players across train/test, and if so, fix it before trusting any new feature or model number.

Player identity is a two-sided problem here: every row has a `white_id` *and* a `black_id`, and both need to stay out of the opposite split. A standard `GroupShuffleSplit` only takes one group column per row, so before picking a split strategy we first need to know the shape of the player graph (nodes = players, edges = games) — specifically, whether it's one giant connected component or many small ones.

## 1. Player graph connected components\n\nEvery row has `white_id` and `black_id`. Union-find over (white_id, black_id) edges tells us whether the player population is one giant blob (bad for a clean disjoint split) or many separable pieces.

In [1]:
import pandas as pd
from collections import Counter

df_raw = pd.read_csv("../data/club_games_data.csv", usecols=["white_id", "black_id"])
print(f"rows: {len(df_raw)}")

parent = {}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

for pid in pd.concat([df_raw["white_id"], df_raw["black_id"]]).unique():
    parent[pid] = pid

for w, b in zip(df_raw["white_id"], df_raw["black_id"]):
    union(w, b)

roots = Counter(find(pid) for pid in parent)
sizes = sorted(roots.values(), reverse=True)

print(f"total players (nodes): {len(parent)}")
print(f"total components: {len(sizes)}")
print(f"largest component: {sizes[0]} players ({100*sizes[0]/len(parent):.2f}% of all players)")
print(f"top 10 component sizes: {sizes[:10]}")
print(f"components of size 1: {sum(1 for s in sizes if s == 1)}")
print(f"components of size >= 10: {sum(1 for s in sizes if s >= 10)}")
print(f"components of size >= 100: {sum(1 for s in sizes if s >= 100)}")

giant_root = roots.most_common(1)[0][0]
giant_players = {pid for pid in parent if find(pid) == giant_root}
rows_in_giant = df_raw["white_id"].isin(giant_players) | df_raw["black_id"].isin(giant_players)
print(f"rows (games) touching giant component: {rows_in_giant.sum()} / {len(df_raw)} = {100*rows_in_giant.mean():.2f}%")

rows: 66879


total players (nodes): 56234
total components: 1168
largest component: 45343 players (80.63% of all players)
top 10 component sizes: [45343, 290, 257, 179, 144, 117, 115, 115, 111, 87]
components of size 1: 0
components of size >= 10: 279
components of size >= 100: 9
rows (games) touching giant component: 55552 / 66879 = 83.06%


**Finding**: one giant component holds 45,343 players (80.6% of all players) and touches 55,552 games (83.1% of rows). The other 1,167 components are small (max 290 players). This rules out a naive per-row `GroupShuffleSplit` on a single id column, but a clean split is still achievable: since the giant component can't be split, it goes entirely on one side (train), and the remaining ~17% of games (all in small, disjoint components) get bin-packed into train/test to land close to an 80/20 ratio. Test set will necessarily be drawn only from the long-tail small-component population — worth flagging as a caveat on any test-set number downstream, since it's not a uniform random sample of players.

## 2. Player-disjoint split

Since every game's `white_id`/`black_id` union into the same component by construction, each row belongs to exactly one component — there's no ambiguity to resolve. The giant component (83.06% of rows) can't be split, so it goes entirely to train; every other (small, disjoint) component goes to test. That caps test size at 16.94% of rows rather than a clean 20% — the giant component leaves only 11,327 rows to work with — but it's the largest test set achievable without leaking a player across the split.

Also regenerating `features/pgn_features.csv` as `features/pgn_features_v3.csv` with explicit `white_id`/`black_id` columns, replacing the old positional-only join (`elosense_v2.ipynb` and its cached file are untouched).

In [2]:
import numpy as np

pgn_feats = pd.read_csv("../features/pgn_features.csv")
assert len(pgn_feats) == len(df_raw), "row count mismatch — positional join invalid"

pgn_feats_v3 = pd.concat([df_raw.reset_index(drop=True), pgn_feats.reset_index(drop=True)], axis=1)

train_mask = rows_in_giant.reset_index(drop=True)
test_mask = ~train_mask
print(f"train rows: {train_mask.sum()} ({100*train_mask.mean():.2f}%)")
print(f"test rows: {test_mask.sum()} ({100*test_mask.mean():.2f}%)")

train_players = set(df_raw.loc[train_mask, "white_id"]) | set(df_raw.loc[train_mask, "black_id"])
test_players = set(df_raw.loc[test_mask, "white_id"]) | set(df_raw.loc[test_mask, "black_id"])
overlap = train_players & test_players
print(f"player overlap between train/test: {len(overlap)}")
assert len(overlap) == 0, "split is leaky!"

pgn_feats_v3["split"] = np.where(train_mask, "train", "test")
pgn_feats_v3.to_csv("../features/pgn_features_v3.csv", index=False)
print("saved features/pgn_features_v3.csv, shape:", pgn_feats_v3.shape)
pgn_feats_v3.head()

train rows: 55552 (83.06%)
test rows: 11327 (16.94%)


player overlap between train/test: 0


saved features/pgn_features_v3.csv, shape: (66879, 11)


,white_id,black_id,white_rating,black_rating,eco,opening_moves,ply_count,max_swing,lead_changes,white_avg_move_time,split
0,https://api.chess.com/pub/player/-amos-,https://api.chess.com/pub/player/miniman2804,1708,1608,E22,d4 Nf6 c4 e6 Nc3 Bb4 Qb3 Bxc3+ Qxc3 O-O,43,13,2,NaN,test
1,https://api.chess.com/pub/player/-amos-,https://api.chess.com/pub/player/koltcho69,1726,1577,C53,e4 e5 Nf3 Nc6 Bc4 Bc5 c3 a6 d4 exd4,65,16,9,NaN,test
2,https://api.chess.com/pub/player/-amos-,https://api.chess.com/pub/player/enhmandah,1727,842,D00,d4 d5 e3 c6 c4 dxc4 Bxc4 b5 Bb3 a5,29,2,1,NaN,test
3,https://api.chess.com/pub/player/enhmandah,https://api.chess.com/pub/player/-amos-,819,1727,B20,e4 c5 b3 Nc6 a4 d6 Bb5 Bd7 Qf3 Nd4,32,13,0,NaN,test
4,https://api.chess.com/pub/player/-amos-,https://api.chess.com/pub/player/shalllow-blue,1729,1116,A40,d4 e6 c4 Qf6 Nf3 d6 Bg5 Qg6 Nc3 c6,45,11,2,NaN,test
